In [ ]:
from tda_module.v3_1 import CloudDataManager

manager = CloudDataManager()
# manager.load_groups_from_directory("/workspaces/Cellular-TDA/data/Data 8:11")

# manager.TDA.compute_all_persistence_diagrams()
# manager.TDA.compute_all_persistence_landscapes(n_layers=2,resolution=500)


manager._tda_params

In [ ]:
path = "../notebooks/currentData.pkl"
# manager.save(path)  # Run after loading data and computing landscapes
manager = CloudDataManager.load(path)
print(len(manager.groups))

In [ ]:
plot1 = {
    "embryo_name": "All",
    "condition": [ "control-m", "epha2a-wm"],  
    "properties": [0]
}
plot2 = {
    "embryo_name": "All",
    "condition": [ "control-m", "epha2a-wm"],  
    "properties": [1]
}
plot3 = {
    "embryo_name": "All",
    "condition": [ "control-l", "epha2a-wl"],
    "properties": [0]
}
plot4 = {
    "embryo_name": "All",
    "condition": [ "control-l", "epha2a-wl"],
    "properties": [1]
}
plot5 = {
    "embryo_name": "All",
    "condition": [ "control-m", "control-l" ],
    "properties": [0]
}
plot6 = {
    "embryo_name": "All",
    "condition": [ "control-m", "control-l" ],
    "properties": [1]
}
plot7 = {
    "embryo_name": "All",
    "condition": [ "control-m", "control-l", "epha2a-wl" ],
    "properties": [0]
}   
plot8 = {
    "embryo_name": "All",
    "condition": [ "control-m", "control-l", "epha2a-wl" ],
    "properties": [ 1]
}

from plotly.subplots import make_subplots

all_plots = [plot1, plot2, plot3, plot4, plot5, plot6, plot7, plot8]
features = {
    "marker_symbols": {"CM1": "circle"},
    "markers": {"size": 10},
    "condition_colors": {
        "control-l": "#260eff",
        "control-m": "#260eff",
        "epha2a-wm": "#ff0e16",
        "epha2a-wl": "#ff0e16",
    },
    "group_colors": {"CL1": "#260eff"},
}

fig = make_subplots(
    rows=4,
    cols=2,
    vertical_spacing=0.04,
    subplot_titles=[f"Plot {i}" for i in range(1, 9)],
)
for annotation, pca_key in zip(fig.layout.annotations, all_plots):
    conditions = pca_key["condition"]
    if isinstance(conditions, str):
        conditions = [conditions]
    properties = pca_key["properties"]
    annotation.text = (
        f"{' + '.join(conditions)} | "
        f"H{', '.join(str(prop) for prop in properties)}"
    )

for i, pca_key in enumerate(all_plots):
    pca_result = manager.compute_pca(pca_key)
    plot_fig = manager.plot_interactive_plotly(
        pca_key,
        showTrajectories=False,
        features=features,
    )

    row = i // 2 + 1
    col = i % 2 + 1
    legend_id = "legend" if i == 0 else f"legend{i + 1}"
    for trace in plot_fig.data:
        trace.legend = legend_id
        fig.add_trace(trace, row=row, col=col)

    # Keep each subplot's PCA bounds fixed when legend entries are hidden.
    fig.update_xaxes(
        range=plot_fig.layout.xaxis.range,
        title_text=(
            f"PC1 ({pca_result['explained_variance_ratio'][0] * 100:.2f}%)"
        ),
        autorange=False,
        row=row,
        col=col,
    )
    fig.update_yaxes(
        range=plot_fig.layout.yaxis.range,
        title_text=(
            f"PC2 ({pca_result['explained_variance_ratio'][1] * 100:.2f}%)"
        ),
        autorange=False,
        row=row,
        col=col,
    )

# Put an independent legend in the bottom band of each subplot.
legend_band = 0.035
for i in range(len(all_plots)):
    axis_number = i + 1
    suffix = "" if axis_number == 1 else str(axis_number)
    xaxis = fig.layout[f"xaxis{suffix}"]
    yaxis = fig.layout[f"yaxis{suffix}"]

    x0, x1 = xaxis.domain
    y0, y1 = yaxis.domain
    legend_y = y0 + legend_band
    yaxis.domain = [legend_y, y1]

    legend_id = "legend" if i == 0 else f"legend{i + 1}"
    fig.update_layout(**{
        legend_id: dict(
            orientation="h",
            x=(x0 + x1) / 2,
            xanchor="center",
            y=legend_y,
            yanchor="top",
            font=dict(size=9),
        )
    })

fig.update_layout(height=1600, width=1200, title_text="PCA Plots")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(4, 2, figsize=(18, 24))
for ax, pca_key in zip(axes.flat, all_plots):
    manager.compute_pca(pca_key)
    plot_fig = manager.plot_sequential_vectors(pca_key)

    # Compose the standalone plot into this grid cell without changing its API.
    plot_fig.canvas.draw()
    ax.imshow(np.asarray(plot_fig.canvas.buffer_rgba()))

    conditions = pca_key["condition"]
    if isinstance(conditions, str):
        conditions = [conditions]
    properties = pca_key["properties"]
    ax.set_title(
        f"{' + '.join(conditions)} | "
        f"H{', '.join(str(prop) for prop in properties)}"
    )
    ax.axis("off")
    plt.close(plot_fig)

fig.suptitle("Sequentially Reoriented PCA Trajectories", fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.98))
plt.show()